# Gatekeeper triage, end to end

Strands Decider 2B takes a state and a list of typed questions (a choice among options, a yes/no, or a score) and
returns, for each, an answer with a probability. The gatekeeper asks every question about a task in **one read**:
six typed questions, one round trip. Below a floor, the host treats the answer as "unsure", and **unsure is a
stop**: if the step that would decide the outcome rests on an answer below its floor, the host stops, applies the
project default, and records which question was unsure. It never reads an unsure yes/no as a no.

This notebook walks that pattern on one task and two short transcripts from the repository's fictional fixtures.
It needs no keys and makes no Bedrock call: the extraction step (an open-weight model on Amazon Bedrock turning a
transcript into tasks) is described in the [README](../README.md), not run here. With a Strands Decider server on
this machine it asks it live; without one it replays the answers Decider gave in the repository's bench. More in
the README and in the article [*Ratatoskr: A Smart Wearable Dispatcher, Powered by Strands Decider 2B*](https://builder.aws.com/project/3KyHho3TSCx8dWc3u14ypmOfgXs).
A screen-capture walkthrough of the courier running on three fictional recordings is at https://youtu.be/RGEm62mWekI.

## 1. Setup

Installs the repository (with its `pattern` extra) only if it is not importable yet, then imports the pattern from
`patterns/gatekeeper_triage/`, which is standard-library Python.

In [1]:
import copy, importlib.util, json, os, platform, subprocess, sys, time, urllib.request
from importlib.metadata import version
from pathlib import Path
from IPython.display import Markdown, display

REPO = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").exists() and (p / "patterns" / "gatekeeper_triage").exists())
try:
    import ratatoskr_demo  # already installed: a second run does not reinstall
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[pattern]"], check=True)
    import ratatoskr_demo

sys.path.insert(0, str(REPO / "patterns" / "gatekeeper_triage"))
import decider_client as gatekeeper

print(f"Python {platform.python_version()}, ratatoskr-demo {version('ratatoskr-demo')}")
print("strands-decider:", version("strands-decider") if importlib.util.find_spec("strands_decider")
      else "not installed in this environment")
print("pattern: patterns/gatekeeper_triage/decider_client.py (standard library)")

Python 3.12.10, ratatoskr-demo 0.1.0
strands-decider: not installed in this environment
pattern: patterns/gatekeeper_triage/decider_client.py (standard library)


## 2. Find a Decider

Probes `/health` on `127.0.0.1:8000`. If nothing answers and `strands-decider` is installed in this environment,
it starts `tools/serve_decider.py` and waits up to two minutes; otherwise the notebook replays recorded answers.

On SageMaker, a CPU instance with 16 GB of memory works. To let this cell start the server, install
`strands-decider==0.1.0` into the notebook's environment first; the first start downloads 4.4 GB of weights,
which can take longer than the two minutes this cell waits, so run it again once the download is done.

In [2]:
DECIDER_URL = os.environ.get("DECIDER_URL", "http://127.0.0.1:8000")
START_DECIDER = True  # set False to never start a server from here


def healthy(timeout: float = 3.0) -> bool:
    try:
        with urllib.request.urlopen(f"{DECIDER_URL}/health", timeout=timeout) as reply:
            return reply.status == 200
    except OSError:
        return False


LIVE = healthy()
if not LIVE and START_DECIDER and importlib.util.find_spec("strands_decider") and DECIDER_URL.endswith(":8000"):
    (REPO / "ledger").mkdir(exist_ok=True)
    server = subprocess.Popen([sys.executable, str(REPO / "tools" / "serve_decider.py"),
                               "--pid-file", str(REPO / "ledger" / "decider.pid")], cwd=REPO,
                              stdout=open(REPO / "ledger" / "decider-serve.log", "w"), stderr=subprocess.STDOUT)
    deadline = time.time() + 120
    while time.time() < deadline and not healthy():
        time.sleep(3)
    LIVE = healthy()
print("Mode:", "LIVE, Strands Decider answering on this machine" if LIVE
      else "RECORDED, the answers Strands Decider gave in the bench pass of 19 Oct 2026")

Mode: LIVE, Strands Decider answering on this machine


## 3. The questions

`questions.toml` holds the six questions Decider is asked, each with its kind and its floor. The floors stay on
this side: Decider returns probabilities, and the host decides what is sure.

In [3]:
questions = gatekeeper.load_questions()
rows = ["| question | kind | floor | what it asks |", "|---|---|---|---|"]
for name in gatekeeper.QUESTION_NAMES:
    entry = questions.entries[name]
    rows.append(f"| `{name}` | {entry['type']} | {questions.floor(name):.2f} | {entry['instructions'].replace('|', '/')} |")
display(Markdown("\n".join(rows)))

| question | kind | floor | what it asks |
|---|---|---|---|
| `project` | choice | 0.70 | Which of the owner's projects or areas does this task belong to? Judge from the task text, the notes, and the contact's organisation and role. Choose 'other' if none fits well. |
| `someone_waiting` | noul | 0.70 | Is a specific person — named, or identified by relationship or role — expecting a reply or a deliverable from the owner, whether or not a date was given? It is not enough that the owner is waiting on someone else. |
| `hard_deadline` | noul | 0.70 | Is there a date, time, or event set by someone other than the owner, or agreed with them, by which this must be done, after which it is late or the chance is gone? A date the owner picked for themself is not one. |
| `worth_researching_soon` | noul | 0.70 | Is this worth investigating or following up on in the next few weeks even though nobody is waiting on a reply by a date? |
| `real_commitment` | noul | 0.70 | Is this a real commitment that someone actually made, rather than a pleasantry? |
| `sensitive` | noul | 0.60 | Does this task or its notes contain details that should not go into a shared task list before the owner has reviewed them? |

## 4. One task

A task from the bench fixtures (fictional), with the timestamp of the recording it came from. The state is what
the pattern shows Decider: the task (content, notes, due string) and the contact line. Never the transcript.

In [4]:
from ratatoskr_demo import due
from ratatoskr_demo.expected import canonical_tasks, load_expected
from ratatoskr_demo.route import routing_state

RECORDING = "rec_010"
fixture = json.loads(next((REPO / "bench" / "fixtures").glob(f"{RECORDING}_*.json")).read_text(encoding="utf-8"))
recording_at = fixture["data"]["recording_at"]
expected_task, contact = next((e, c) for rid, e, c in canonical_tasks(load_expected())
                              if rid == RECORDING and e.task.content.startswith("Send Wren"))
task = expected_task.task
state = {"task": {"content": task.content, "notes": task.notes, "due_string": task.due_string},
         "contact": {"name": contact.name, "organisation": contact.organisation, "role": contact.role,
                     "context": contact.context}}
assert state == routing_state(task, contact)  # the same state the courier sends for this task

deadline_near = due.near(task.due_string, due.recording_date(recording_at))
print(f"recording {RECORDING}, recorded at {recording_at}")
print(f"due string {task.due_string!r}: within 30 days of the recording, read in code: {deadline_near}")
print(json.dumps(state, indent=2, ensure_ascii=False))

recording rec_010, recorded at 2026-10-05T19:30:00Z
due string 'end of the month': within 30 days of the recording, read in code: True
{
  "task": {
    "content": "Send Wren a first draft of the ground stations article (about 1,200 words)",
    "notes": null,
    "due_string": "end of the month"
  },
  "contact": {
    "name": "Wren Castellanos",
    "organisation": "Lowlands Science Weekly",
    "role": "newsletter editor",
    "context": "Coffee with a newsletter editor who asked for an article on amateur ground stations."
  }
}


## 5. One read

All six questions in one request. Live, the answers come from the server; recorded, they are the answers Decider
gave for this same task in the bench pass (`bench/recorded_answers.json`, exported from the raw per-task file
`bench/RESULTS.md` is rendered from). Either way the next cells read the same structure.

In [5]:
recorded = json.loads((REPO / "bench" / "recorded_answers.json").read_text(encoding="utf-8"))


class RecordedDecider(gatekeeper.DeciderClient):
    """Replays answers Strands Decider gave in the bench pass, one per request, so no server is needed."""

    def __init__(self, responses):
        super().__init__(DECIDER_URL)
        self.responses = list(responses)

    def systemone(self, state, questions):
        return self.responses.pop(0)


if LIVE:
    started = time.perf_counter()
    response = gatekeeper.DeciderClient(DECIDER_URL).systemone(state, questions.request_questions())
    seconds = time.perf_counter() - started
else:
    response = recorded["triage"]["response"]
    seconds = response["latency_ms"] / 1000

rows = ["| question | answer | probability of yes | confidence | floor | sure |", "|---|---|---|---|---|---|"]
for name in gatekeeper.QUESTION_NAMES:
    answer = response["answers"].get(name) or {}
    floor = questions.floor(name)
    if "choice" in answer:
        confidence = answer["confidence"]
        rows.append(f"| `{name}` | {answer['choice']} | | {confidence:.2f} | {floor:.2f} | "
                    f"{'yes' if confidence >= floor else 'no'} |")
    elif "noul" in answer:
        p = answer["noul"]
        rows.append(f"| `{name}` | {'yes' if p >= 0.5 else 'no'} | {p:.2f} | {max(p, 1 - p):.2f} | {floor:.2f} | "
                    f"{'yes' if max(p, 1 - p) >= floor else 'no'} |")
    else:
        rows.append(f"| `{name}` | no usable answer | | | {floor:.2f} | no |")
display(Markdown("\n".join(rows)))
print(f"LIVE: one read, six answers, {seconds:.1f} s" if LIVE else
      f"RECORDED: one read, six answers, {seconds:.1f} s when the bench asked it on the reference laptop's CPU")
if LIVE:
    rounded = lambda r: {k: round(v.get("noul", v.get("confidence", 0)), 2) for k, v in r["answers"].items()}
    print("same as the bench's recorded answers for this task:",
          rounded(response) == rounded(recorded["triage"]["response"]))

| question | answer | probability of yes | confidence | floor | sure |
|---|---|---|---|---|---|
| `project` | writing | | 0.87 | 0.70 | yes |
| `someone_waiting` | yes | 0.89 | 0.89 | 0.70 | yes |
| `hard_deadline` | no | 0.26 | 0.74 | 0.70 | yes |
| `worth_researching_soon` | no | 0.39 | 0.61 | 0.70 | no |
| `real_commitment` | yes | 0.91 | 0.91 | 0.70 | yes |
| `sensitive` | no | 0.17 | 0.83 | 0.60 | yes |

LIVE: one read, six answers, 10.3 s
same as the bench's recorded answers for this task: False


## 6. The rule

The answers become actions in code. Priority walks its steps in order: someone waiting means high; a hard
deadline within thirty days means high; worth researching soon means medium; otherwise low. The project is the
choice if Decider is sure of it, else the task is filed unsorted.

In [6]:
triage = gatekeeper.judge(response, questions, deadline_near=deadline_near)
print(f"verdict {triage.verdict}; file under {triage.project or 'unsorted'}; "
      f"priority {triage.priority or '-'}, set by {triage.priority_source}")

forced = copy.deepcopy(response)
forced["answers"]["someone_waiting"] = {"type": "noul", "noul": 0.55}
again = gatekeeper.judge(forced, questions, deadline_near=deadline_near)
print(f"someone_waiting forced to p 0.55 (confidence 0.55, under its {questions.floor('someone_waiting'):.2f} floor): "
      f"priority {again.priority or '-'}, set by {again.priority_source}")

verdict proceed; file under writing; priority high, set by someone_waiting
someone_waiting forced to p 0.55 (confidence 0.55, under its 0.70 floor): priority low, set by default:waiting


With the waiting question under its floor, the rule does not move on to the deadline question as if the answer
were no. It stops at the step that would have decided, takes the project's default priority, and records which
question was unsure: `default:waiting`. Unsure is a stop, never a no.

## 7. The privacy gate

Before any cloud model sees a word, the gate reads the transcript itself, in windows of about 300 words, and asks
the `sensitive` question of each. Sensitive or unsure in any window holds the recording. Two fictional transcripts
from the fixtures: one ordinary, one about a colleague's medical leave. The notebook reads them from the fixture
files and never prints them.

In [7]:
from ratatoskr_demo.transcript import extract_text

ABOUT = {"rec_001": "a networking chat at a meetup", "rec_014": "a hallway conversation about a colleague's medical leave"}
rows = ["| recording | what it is | words | windows asked | highest p(sensitive) | gate |", "|---|---|---|---|---|---|"]
for rid in ("rec_001", "rec_014"):
    path = next(p for folder in ("bench/fixtures", "tests/fixtures") for p in (REPO / folder).glob(f"{rid}_*.json"))
    text = extract_text(json.loads(path.read_text(encoding="utf-8"))["data"])
    client = gatekeeper.DeciderClient(DECIDER_URL) if LIVE else RecordedDecider(recorded["gate"][rid]["windows"])
    verdict = client.gate_source_text(text, questions)
    rows.append(f"| {rid} | {ABOUT[rid]} | {len(text.split())} | {verdict.windows_asked} of {verdict.windows_total} | "
                f"{verdict.max_p:.2f} | **{verdict.verdict}**{f' ({verdict.reason})' if verdict.reason else ''} |")
display(Markdown("\n".join(rows)))

| recording | what it is | words | windows asked | highest p(sensitive) | gate |
|---|---|---|---|---|---|
| rec_001 | a networking chat at a meetup | 144 | 1 of 1 | 0.19 | **proceed** |
| rec_014 | a hallway conversation about a colleague's medical leave | 82 | 1 of 1 | 0.85 | **hold** (source-sensitive) |

An unreadable or missing answer holds or waits; it never passes. Here, if Decider cannot be asked or its
`sensitive` answer cannot be read, the gate returns `hold (decider-unavailable)` (`questions.toml` sets
`on_unavailable = "hold"`). In the courier the recording waits for a healthy server, and nothing is extracted.

## 8. The bench

The repository's numbers, read from `bench/RESULTS.md` rather than typed in: on the thirty canonical tasks, how
often each answerer was sure, and how often it was right when sure.

In [8]:
results = (REPO / "bench" / "RESULTS.md").read_text(encoding="utf-8")
section = results.split("## Agreement on the canonical tasks (no extraction)")[1].split("\n## ")[0]
table = {}
for line in section.splitlines():
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    if len(cells) == 5 and cells[0] in ("decider", "bedrock", "jev") and cells[3]:
        table[(cells[0], cells[1])] = (cells[3], cells[4])
names = {"decider": "Strands Decider", "bedrock": "gpt-oss-20b", "jev": "Jev"}
rows = ["| question | " + " | ".join(f"{n} sure | {n} right when sure" for n in names.values()) + " |",
        "|---|" + "---|---|" * len(names)]
for q in dict.fromkeys(q for _, q in table):
    rows.append(f"| {q} | " + " | ".join(f"{table[(b, q)][0]} | {table[(b, q)][1]}" for b in names) + " |")
display(Markdown("\n".join(rows)))
print(results.splitlines()[3])

| question | Strands Decider sure | Strands Decider right when sure | gpt-oss-20b sure | gpt-oss-20b right when sure | Jev sure | Jev right when sure |
|---|---|---|---|---|---|---|
| project | 18/30 (60%) | 18/18 (100%) | 28/30 (93%) | 24/28 (86%) | 26/30 (87%) | 25/26 (96%) |
| priority | 12/30 (40%) | 9/12 (75%) | 28/30 (93%) | 20/28 (71%) | 22/30 (73%) | 17/22 (77%) |
| someone_waiting | 15/30 (50%) | 12/15 (80%) | 29/30 (97%) | 20/29 (69%) | 18/30 (60%) | 17/18 (94%) |
| hard_deadline | 21/30 (70%) | 16/21 (76%) | 29/30 (97%) | 17/29 (59%) | 25/30 (83%) | 15/25 (60%) |
| worth_researching_soon | 3/11 (27%) | 1/3 (33%) | 9/11 (82%) | 5/9 (56%) | 2/11 (18%) | 1/2 (50%) |
| real_commitment | 25/30 (83%) | 25/25 (100%) | 29/30 (97%) | 28/29 (97%) | 26/30 (87%) | 26/26 (100%) |
| sensitive | 30/30 (100%) | 30/30 (100%) | 28/30 (93%) | 27/28 (96%) | 29/30 (97%) | 29/29 (100%) |

Generated 2026-10-20 00:15 UTC by `bench/run_bench.py` (git blob `f2d4beeeb0f3509e`); re-rendered 2026-10-21 05:13 UTC from the run's saved state with the rotation below, no model asked again.


## 9. The fresh evaluation

Twenty-four new fictional conversations, twelve sensitive and twelve ordinary, labelled before any model was asked
about them, run once through the courier on the frozen configuration (`eval-freeze-2026-10-22`). Read from
`bench/FRESH_EVAL_2026-10-22.md` as section 8 reads RESULTS.md: first the run's metadata and its counts, then
Decider's numbers beside the baseline's on the questions both runs ask.

In [9]:
fresh = (REPO / "bench" / "FRESH_EVAL_2026-10-22.md").read_text(encoding="utf-8")
head = fresh.split("\n## ")[0].splitlines()
display(Markdown("\n".join(line for line in head if line.startswith(("Generated", "- **", "| ")))))
counts = fresh.split("## The counts")[1].split("\n## ")[0].split("\nSources:")[0]
display(Markdown(counts.strip()))

Generated 2026-10-21 05:10 UTC by `bench/run_bench.py --fresh` (git blob `f2d4beeeb0f3509e`); run 2026-10-21 04:57 UTC to 2026-10-21 05:10 UTC.
- **Code revision:** tag `eval-freeze-2026-10-22` (commit `9b76c4e`). The courier's code, prompts, floors and rules are those of the tag (no difference under `ratatoskr_demo/` or in `config.toml`).
- **Questions and labels:** the question wording and the labelling conventions are those of `eval-freeze-2026-10-22`. The 18 Oct bench's original labels are preserved in RESULTS.md's "Label corrections" section.
- **The fresh labels:** `bench/fixtures/fresh/labels.json`, sha256 `87342167a95bf228 f13844fa61bc4ffd 74ab4a123ceec400 d0b094fe69498d77` (in groups of sixteen: the repository's secret scan reads any longer hex run as a key), committed in `896d680` before the first model call, and not edited after the run.
- **Not in this set:** the five recordings made on 21 Oct for the walkthrough video are in no count here.
| | |
| extraction model | `openai.gpt-oss-20b` on Bedrock (bedrock-mantle, us-east-1), reasoning effort `low`, max_tokens 2000 |
| Strands Decider | `StrandsAgents/strands-decider-2B-hobson-v21` on cpu, strands-decider 0.1.0, prefix cache on |
| configuration | `config.toml` git blob `bc31acc3100b03eb`; `ratatoskr_demo/prompts/route.toml` git blob `746b41993c51ad3b` |
| hardware | the build laptop's CPU (16 logical cores), Decider capped at 4 threads at below-normal priority, 600 s per read at most; kernels: reference PyTorch implementation (causal_conv1d and flash-linear-attention not installed; CPU-only torch build) |
| conditions | another long-running Bedrock experiment on the same laptop paused; the laptop otherwise idle |

| count | result |
|---|---|
| sensitive conversations wrongly cleared (written to Todoist) | 0 of 12 |
| sensitive conversations cleared by the source gate and held after extraction | 0 of 12 |
| ordinary conversations unnecessarily held: source gate | 3 of 12 |
| ordinary conversations unnecessarily held: tripwire | 0 of 12 |
| ordinary conversations unnecessarily held: task check | 1 of 12 |
| Bedrock extraction calls for held conversations | 1 (over 16 held) |
| task writes for held conversations | 0 (over 16 held) |
| project, labelled ordinary tasks: correct automatic assignments | 8 of 26 |
| project: incorrect automatic assignments | 0 of 26 |
| project: abstentions (left unsorted) | 18 of 26 |
| priority under ruling 7 (policy A), labelled ordinary tasks: right | 11 of 26 |
| priority: wrong | 15 of 26 |
| priority: set by the rule (not the project's default), and right of those | 14 of 26; 9 of 14 |
| source-gate read (one transcript window), median / p90 latency | 2,070 ms / 4,121 ms over 26 reads |
| task read (six questions), median / p90 latency | 11,123 ms / 11,286 ms over 63 reads |

In [10]:
def decider_rows(text, heading, label):
    """(question) -> (sure, right when sure) for one answerer's rows of one table."""
    section = text.split(heading)[1].split("\n## ")[0]
    rows = {}
    for line in section.splitlines():
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        if len(cells) == 5 and cells[0].startswith(label) and cells[3]:
            rows[cells[1]] = (cells[3], cells[4])
    return rows


baseline = decider_rows(results, "## Agreement on the canonical tasks (no extraction)", "decider")
fresh_rows = decider_rows(fresh, "## Every question on the labelled tasks", "ordinary")
shared = [q for q in baseline if q in fresh_rows]
rows = ["| question | baseline: sure | baseline: agree when sure | fresh: sure | fresh: agree when sure |",
        "|---|---|---|---|---|"]
rows += [f"| {q} | {baseline[q][0]} | {baseline[q][1]} | {fresh_rows[q][0]} | {fresh_rows[q][1]} |" for q in shared]
display(Markdown("\n".join(rows)))
print("baseline:", results.splitlines()[3])
print("fresh:   ", fresh.splitlines()[3])

| question | baseline: sure | baseline: agree when sure | fresh: sure | fresh: agree when sure |
|---|---|---|---|---|
| project | 18/30 (60%) | 18/18 (100%) | 8/26 (31%) | 8/8 (100%) |
| priority | 12/30 (40%) | 9/12 (75%) | 14/26 (54%) | 9/14 (64%) |
| someone_waiting | 15/30 (50%) | 12/15 (80%) | 13/26 (50%) | 8/13 (62%) |
| hard_deadline | 21/30 (70%) | 16/21 (76%) | 13/26 (50%) | 9/13 (69%) |
| worth_researching_soon | 3/11 (27%) | 1/3 (33%) | 3/10 (30%) | 0/3 (0%) |
| real_commitment | 25/30 (83%) | 25/25 (100%) | 24/26 (92%) | 23/24 (96%) |
| sensitive | 30/30 (100%) | 30/30 (100%) | 24/26 (92%) | 24/24 (100%) |

baseline: Generated 2026-10-20 00:15 UTC by `bench/run_bench.py` (git blob `f2d4beeeb0f3509e`); re-rendered 2026-10-21 05:13 UTC from the run's saved state with the rotation below, no model asked again.
fresh:    Generated 2026-10-21 05:10 UTC by `bench/run_bench.py --fresh` (git blob `f2d4beeeb0f3509e`); run 2026-10-21 04:57 UTC to 2026-10-21 05:10 UTC.


The baseline is the bench pass of 19 Oct on the thirty canonical tasks (labels as corrected on 20 Oct); the fresh
column is Decider on the labelled tasks of the twelve ordinary conversations, asked as written. Both are Strands
Decider on the same laptop CPU; the fresh run used the frozen question wording, the baseline the wording of 19 Oct.

## Close

The invariants (`CONTRIBUTING.md`): a change that does any of these is declined.

- Passes an unsure privacy answer. A sensitivity answer below its floor holds; it is never read as a no.
- Lets the `urgent` tag, or anything else, bypass a verdict Decider has given.
- Stores a transcript, or puts recording text into a notice, a log, or the wrist line.
- Binds the Decider server to anything other than the loopback address.
- Raises the spend ceiling in code or lets config raise it past five dollars.
- Weakens a floor without a bench result showing why.

Copyright 2026 Dan Lee-Odinson. Licensed under the Apache License 2.0; see LICENSE and NOTICE.

I directed this build and dispositioned every design question in it; Claude Code wrote the code under that
direction.

In [11]:
if "server" in globals():  # a Decider server this notebook started in section 2: stop it
    server.terminate()
    print("stopped the Decider server this notebook started")